In [1]:
# ================= NOTEBOOK 4: QLORA SFT & DPO REWARD OPTIMIZATION =================
!pip install -q -U bitsandbytes peft accelerate datasets

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Load Qwen2.5-1.5B-Instruct in 4-bit quantization
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16
)
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
sft_model = get_peft_model(base_model, lora_config)
sft_model.print_trainable_parameters()
print("Qwen Model loaded successfully for QLoRA SFT & DPO training!")

ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device



ModuleNotFoundError: No module named 'torch'